# 推理模型工程：从可验证训练到测试时计算

> **本章定位**：整合 `41` 的 SFT/GRPO、`91` 的 CoT/Self-Consistency 与 `60` 的服务契约，形成推理数据、验证训练与测试时计算的工程能力链。

> **章节边界**：本章属于模型训练与适配：推理训练专题。思维链（Chain-of-Thought，CoT）并非独立网络层，也不存在通用的“CoT Loss”；本章聚焦数据、监督粒度、验证器、采样预算与输出治理，不重新推导通用训练或服务基础。

**本章总览**：内容从可验证任务与推理轨迹出发，依次覆盖拒绝采样、结果与过程奖励、经验证的蒸馏、测试时预算分配和受控服务接口。

```mermaid
flowchart LR
    D["问题 + 可验证任务"] --> S["Reasoning-trace SFT"]
    S --> R["ORM / PRM / RLVR"]
    R --> T["Verified Distillation"]
    T --> C["Test-Time Compute"]
    C --> V["Verifier Selection"]
    V --> A["答案 + 可验证证据"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 模型训练与适配：推理训练专题 |
| 本章定位 | 规模化推理数据、验证器、RLVR、蒸馏与测试时计算。 |
| 先修知识 | 掌握 `40`、`41` 的训练目标与恢复、`50` 的评估及 `91` 的测试时策略；服务与输出治理部分结合 `60`、`70`。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 最小机制 CPU 可运行；真实 Rollout/RL 需要训练与推理解耦集群。 |
| 输入 | 问题、推理轨迹、最终答案、步骤标签、验证器 Revision 与预算。 |
| 交付物 | 版本化推理数据、Verifier、策略制品、服务契约和评测报告。 |

### 1.1．学习目标

完成本章后，读者能够构建包含验证证据的推理数据，区分结果奖励、过程奖励与偏好监督，设计可复算的测试时计算预算，并为推理轨迹、验证器和服务输出建立治理边界。


## 2．直觉与输入输出契约

全章采用同一结构化算术样例：

> 输入：从 7 开始，加 5，再乘 2。  
> 结构化轨迹：`[{"op":"add","value":5},{"op":"multiply","value":2}]`。  
> 最终答案：`24`。

```mermaid
flowchart TD
    Q["Prompt"] --> G["候选轨迹"]
    G --> X["结构化解析"]
    X --> P["逐步执行 / 单元测试 / 符号验证"]
    P --> O["Outcome Reward"]
    P --> M["Process Labels"]
    O --> L["RLVR / 拒绝采样"]
    M --> L
    L --> E["策略与 Verifier 评测"]
```

自然语言轨迹可以帮助训练与搜索，但不能自动成为事实证明或审计记录。生产审计依赖输入、工具调用、执行结果、权限决策和最终输出。


In [ ]:
from dataclasses import dataclass
from typing import Literal

import torch
import torch.nn.functional as F
from pydantic import BaseModel, ConfigDict, Field


@dataclass(frozen=True)
class MyReasoningStep:
    """表示可验证推理轨迹中的一次算术操作。"""
    operation: Literal["add", "multiply"]
    value: int


@dataclass(frozen=True)
class MyReasoningCandidate:
    """保存候选标识、结构化步骤和声明的最终答案。"""
    candidate_id: str
    steps: tuple[MyReasoningStep, ...]
    final_answer: int


def my_execute_steps(initial_value: int, steps: tuple[MyReasoningStep, ...]) -> tuple[list[int], int]:
    """从初值确定性执行步骤并返回完整状态轨迹与最终值。"""
    states = [initial_value]
    current = initial_value
    for step in steps:
        current = current + step.value if step.operation == "add" else current * step.value
        states.append(current)
    return states, current


REFERENCE_STEPS = (
    MyReasoningStep("add", 5),
    MyReasoningStep("multiply", 2),
)
reference_states, reference_answer = my_execute_steps(7, REFERENCE_STEPS)
{"states": reference_states, "answer": reference_answer}


### 2.1．公式、函数与生产组件的对应关系

| 阶段 | 公式/决策 | 原理实现 | 生产库 |
|---|---|---|---|
| Trace SFT | $L=-\sum_t m_t\log p_\theta(y_t\mid y_{<t},x)$ | `my_masked_causal_lm_loss` | `torch.nn.functional.cross_entropy`；TRL `SFTTrainer` |
| 结果奖励 | $r=\mathbb{1}[\hat y=y]$ | `my_verify_candidate` | TRL `GRPOTrainer` 的可组合 Reward Function |
| 组内优势 | $A_i=(r_i-\mu_r)/(\sigma_r+\epsilon)$ | `my_group_relative_advantages` | TRL `GRPOTrainer` |
| Self-Consistency | $\hat y=\arg\max_y\sum_i\mathbb{1}[y_i=y]$ | `my_consensus_answer` | `transformers.generate(num_return_sequences=N)` 或 vLLM 多样本 |
| Best-of-N | $\hat z=\arg\max_{z_i}V(z_i)$ | `my_select_verified_candidate` | vLLM 批量采样 + 独立 Verifier 服务 |

“公式 → 函数 → 库”的对齐还需检查 Label Mask、Reduction、Padding、长度归一化、随机采样和数值精度；函数名相同不代表默认语义相同。


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

推理模型工程把结果正确性、过程证据和计算预算分开建模。一个可审计的候选评分可写为：

$$
S(z)=w_oR_{\mathrm{outcome}}(z)+w_pR_{\mathrm{process}}(z)-\lambda C(z),
\qquad z^*=\arg\max_{z\in\mathcal Z_B}S(z)
$$

其中，$z$ 是候选轨迹，$R_{\mathrm{outcome}}$ 是结果验证分，$R_{\mathrm{process}}$ 是步骤验证分，$C(z)$ 是 Token、时间或调用成本，$\mathcal Z_B$ 是预算 $B$ 内的候选集合。Verifier、Parser 和预算控制器分别对应这些数学对象。权重只定义当前决策策略，不能把不可比的评分伪装成统一真值；原始推理轨迹也不应直接作为用户解释或审计日志。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．推理轨迹监督的标签掩码

Answer-only SFT、Trace+Answer SFT 和只监督答案 Token 是三种不同实验。本节以 `-100` 掩码明确哪些位置参与交叉熵；输入 Prompt 与 Padding 不回传损失。


In [ ]:
IGNORE_INDEX = -100


def my_masked_causal_lm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """右移 Causal LM 预测，并忽略 Prompt 区域计算监督交叉熵。"""
    shifted_logits = logits[:, :-1, :].contiguous()
    shifted_labels = labels[:, 1:].contiguous()
    return F.cross_entropy(
        shifted_logits.view(-1, shifted_logits.size(-1)),
        shifted_labels.view(-1),
        ignore_index=IGNORE_INDEX,
    )


# 42 仅固定最小 Logits 夹具；正式结论使用预注册的多个 Seed，并记录模型、后端与采样配置。
torch.manual_seed(42)
# 固定形状：probe_logits.shape = [1, 8, 32]。
probe_logits = torch.randn(1, 8, 32, requires_grad=True)
# 前 3 个位置是 Prompt；后 5 个位置是结构化轨迹和答案。
# 固定形状：probe_labels.shape = [1, 8]。
probe_labels = torch.tensor([[-100, -100, -100, 4, 8, 12, 16, 24]])
loss = my_masked_causal_lm_loss(probe_logits, probe_labels)
loss.backward()
{"loss": float(loss.detach()), "gradient_is_finite": bool(torch.isfinite(probe_logits.grad).all())}


### 3.2．拒绝采样与步骤验证

最终答案正确不代表过程正确。可执行任务先解析成受限操作，再逐步执行；只有格式、每步状态与最终答案都通过的候选才能进入 Verified Dataset。解析器、执行器和规则版本写入数据行。


In [ ]:
# 3 条候选分别覆盖正确过程、错误过程和答案正确但过程错误；它们是固定验证数据，不代表服务预算。
CANDIDATES = (
    MyReasoningCandidate("correct", REFERENCE_STEPS, 24),
    MyReasoningCandidate(
        "wrong_step",
        (MyReasoningStep("add", 4), MyReasoningStep("multiply", 2)),
        22,
    ),
    MyReasoningCandidate(
        "lucky_answer",
        (MyReasoningStep("multiply", 2), MyReasoningStep("add", 5)),
        24,
    ),
)


def my_verify_candidate(
    initial_value: int,
    candidate: MyReasoningCandidate,
    expected_states: list[int],
    expected_answer: int,
) -> dict[str, object]:
    """执行候选步骤并同时评估逐步正确性、答案奖励和完整验证状态。"""
    states, executed_answer = my_execute_steps(initial_value, candidate.steps)
    step_labels = [
        index < len(expected_states) and state == expected_states[index]
        for index, state in enumerate(states)
    ]
    return {
        "candidate_id": candidate.candidate_id,
        "states": states,
        "step_labels": step_labels,
        "declared_matches_execution": candidate.final_answer == executed_answer,
        # Outcome Reward 只核对候选声明的最终答案；过程与声明一致性由完整验证单独负责。
        "outcome_reward": float(candidate.final_answer == expected_answer),
        "fully_verified": (
            states == expected_states
            and candidate.final_answer == executed_answer
            and executed_answer == expected_answer
        ),
    }


verification_rows = [
    my_verify_candidate(7, candidate, reference_states, reference_answer)
    for candidate in CANDIDATES
]
verified_dataset = [row for row in verification_rows if row["fully_verified"]]
{"verification": verification_rows, "accepted": verified_dataset}


#### 3.2.1．结果正确与过程通过的分离证据

学习问题是：只核对最终答案会遗漏哪些过程错误。下图直接读取 `verification_rows`：左侧逐步显示结构化状态是否与 Reference 一致，右侧并列显示 Outcome Reward 与完整验证结果。验收条件是 `correct` 全部通过，`lucky_answer` 虽然结果奖励为 1，却不能进入 Verified Dataset。


In [ ]:
# 使用真实验证器输出呈现逐步状态、结果奖励与最终准入差异。
import matplotlib.pyplot as plt

candidate_ids = [row["candidate_id"] for row in verification_rows]
step_matrix = torch.tensor([row["step_labels"] for row in verification_rows], dtype=torch.float32)
outcome_rewards = [float(row["outcome_reward"]) for row in verification_rows]
fully_verified = [float(row["fully_verified"]) for row in verification_rows]
row_by_id = {row["candidate_id"]: row for row in verification_rows}
if not row_by_id["correct"]["fully_verified"]:
    raise RuntimeError("正确候选未通过完整验证")
if row_by_id["lucky_answer"]["outcome_reward"] != 1.0 or row_by_id["lucky_answer"]["fully_verified"]:
    raise RuntimeError("结果奖励与过程验证的边界未按预期分离")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
image = axes[0].imshow(step_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
for row_index in range(step_matrix.size(0)):
    for step_index in range(step_matrix.size(1)):
        axes[0].text(step_index, row_index, "通过" if step_matrix[row_index, step_index] else "错误", ha="center", va="center")
axes[0].set(
    title="结构化状态逐步验证", xlabel="状态索引", ylabel="候选",
    xticks=range(step_matrix.size(1)), yticks=range(len(candidate_ids)), yticklabels=candidate_ids,
)
x_positions = torch.arange(len(candidate_ids)).numpy()
bar_width = 0.36
axes[1].bar(x_positions - bar_width / 2, outcome_rewards, width=bar_width, color="#56B4E9", label="Outcome Reward")
axes[1].bar(x_positions + bar_width / 2, fully_verified, width=bar_width, color="#E69F00", label="Fully Verified")
axes[1].set(title="最终结果与完整准入不是同一条件", xlabel="候选", ylabel="0/1", xticks=x_positions, xticklabels=candidate_ids, ylim=(0, 1.15))
axes[1].legend()
fig.colorbar(image, ax=axes[0], shrink=0.8, ticks=[0, 1], label="步骤状态")
plt.tight_layout()
plt.show()
print({"accepted_candidates": [row["candidate_id"] for row in verified_dataset], "outcome_passes": sum(outcome_rewards)})


本图展示的是可公开复算的结构化状态，不是模型的原始隐藏推理。完整验证结果依赖当前 Parser、执行器和 Reference Revision；验证器本身可能存在覆盖不足或可被投机的漏洞。三条固定候选只验证拒绝采样的数据契约，不能据此估计真实模型的 pass@k 或推理质量。


### 3.3．结果、偏好、过程与可验证奖励

- **Preference**：整条回答的相对偏好，适合可读性、相关性与风格，不天然等于推理正确。
- **ORM**（Outcome Reward Model）：对最终结果或整条完成打分。
- **PRM**（Process Reward Model）：逐步给出正确、错误或不确定标签。
- **RLVR**（Reinforcement Learning with Verifiable Rewards）：用答案解析、单元测试、符号执行或环境状态产生可验证奖励。

数学与代码优先使用确定性验证器；人类或模型评审用于难以程序化的质量维度。奖励函数必须版本化，并单独监控格式投机、长度偏好、重复步骤和验证器漏洞。


In [ ]:
# 组内 4 个候选可同时呈现正负奖励；增大组规模通常降低方差，但近似线性增加 Rollout 成本和尾延迟。
GROUP_SIZE = 4
# 0.2 是 PPO/GRPO 裁剪起点；减小更新更保守，增大更易不稳，按 KL、clip fraction、Reward 与验证质量联调。
CLIP_EPSILON = 0.2
# 1e-6 只防止零方差组除零；若经常主导分母，应调整采样或奖励设计。
ADVANTAGE_EPSILON = 1e-6


def my_group_relative_advantages(rewards: torch.Tensor, eps: float = ADVANTAGE_EPSILON):
    """在候选组内标准化奖励，生成相对优势。"""
    mean = rewards.mean()
    standard_deviation = rewards.std(unbiased=False)
    return (rewards - mean) / (standard_deviation + eps)


def my_clipped_policy_objective(
    new_log_probabilities: torch.Tensor,
    old_log_probabilities: torch.Tensor,
    advantages: torch.Tensor,
    clip_epsilon: float,
) -> torch.Tensor:
    """计算带概率比裁剪的策略优化目标。"""
    ratio = torch.exp(new_log_probabilities - old_log_probabilities)
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages
    return -torch.minimum(unclipped, clipped).mean()


# [1,0,1,0] 是固定奖励夹具，用于验证组内标准化的正负方向，不构成生产奖励尺度。
rewards = torch.tensor([1.0, 0.0, 1.0, 0.0])
advantages = my_group_relative_advantages(rewards)
old_log_probabilities = torch.log(torch.tensor([0.40, 0.25, 0.30, 0.20]))
new_log_probabilities = torch.log(torch.tensor([0.44, 0.22, 0.33, 0.18]))
objective = my_clipped_policy_objective(
    new_log_probabilities, old_log_probabilities, advantages, CLIP_EPSILON
)
{"advantages": advantages, "objective": float(objective)}


### 3.4．经验证的推理蒸馏

```mermaid
flowchart LR
    T["Pinned Teacher"] --> N["N 条候选"]
    N --> V["答案 + 步骤验证"]
    V --> D["Verified Distillation Dataset"]
    D --> S["Student SFT"]
    S --> E["独立评测与成本门禁"]
```

蒸馏传递的是教师输出分布与解题行为，不是逻辑真理。数据行至少保存教师 Revision、采样参数、Prompt Version、Verifier Revision、最终答案、结构化步骤、拒绝原因和内容哈希。压缩章 `A60_model_compression.ipynb` 聚焦 Logit/Hidden-state 蒸馏，本章则负责经验证的数据生成链。


### 3.5．测试时计算的预算分配

推理阶段可按 Direct Answer → 单轨迹 CoT → Self-Consistency → Best-of-N + Verifier → 自适应预算逐级扩展。固定采样 N 条候选并不适用于所有问题，预算应结合任务难度、置信度与剩余 Deadline 分配。


In [ ]:
from collections import Counter


def my_consensus_answer(candidates: list[MyReasoningCandidate]) -> int:
    """按多数票选择最终答案，并用较小答案确定性打破平票。"""
    normalized_answers = [candidate.final_answer for candidate in candidates]
    counts = Counter(normalized_answers)
    return min(counts, key=lambda answer: (-counts[answer], answer))


def my_select_verified_candidate(
    initial_value: int,
    candidates: list[MyReasoningCandidate],
    expected_answer: int,
) -> MyReasoningCandidate | None:
    """返回步骤执行与预期答案一致的最小 ID 候选。"""
    verified = []
    for candidate in candidates:
        _, executed_answer = my_execute_steps(initial_value, candidate.steps)
        if executed_answer == expected_answer and candidate.final_answer == executed_answer:
            verified.append(candidate)
    return min(verified, key=lambda item: item.candidate_id) if verified else None


consensus = my_consensus_answer(list(CANDIDATES))
selected = my_select_verified_candidate(7, list(CANDIDATES), reference_answer)
{"consensus_answer": consensus, "verified_candidate": selected}


### 3.6．预算化服务接口

服务接口显式声明推理预算与输出策略，默认不返回原始 CoT。


In [ ]:
# 字符、Token、候选与 Deadline 上限共同限制单请求成本；模型上下文、长度分布或 SLO 变化时按截断率和 P95/P99 联调。
class MyReasoningRequest(BaseModel):
    """定义推理服务的模式、Token、候选数和 Deadline 请求边界。"""
    model_config = ConfigDict(extra="forbid")

    # 1…20,000 是字符级入口边界，Tokenize 后仍须执行模型上下文二次限额。
    prompt: str = Field(min_length=1, max_length=20_000)
    reasoning_mode: Literal["off", "auto", "on"] = "auto"
    reasoning_effort: Literal["low", "medium", "high"] = "medium"
    # 512（0…16,384）限制推理 Token；增大会降低截断但提高 KV、延迟与费用。
    max_reasoning_tokens: int = Field(default=512, ge=0, le=16_384)
    # 1,024（1…32,768）限制总 Token；须与推理预算及模型上下文保持一致。
    max_total_tokens: int = Field(default=1_024, ge=1, le=32_768)
    # 30,000 ms（100…300,000）定义服务期限；须与候选数和 Token 上限联合验收。
    deadline_ms: int = Field(default=30_000, ge=100, le=300_000)
    # 默认 1、最大 32；候选数增加可能提高 pass@k，但 Token/GPU 秒与验证成本近似线性增加。
    candidate_count: int = Field(default=1, ge=1, le=32)
    selection_policy: Literal["greedy", "consensus", "verifier"] = "greedy"
    return_reasoning_summary: bool = False


request = MyReasoningRequest(
    prompt="从 7 开始，加 5，再乘 2。",
    reasoning_mode="on",
    candidate_count=5,  # 5 条候选仅用于呈现 Best-of-N 预算增长；按单位正确答案成本决定是否扩容。
    selection_policy="verifier",
)
request.model_dump()


## 4．证据验证

| 维度 | 建议报告的证据 |
|---|---|
| 正确性 | pass@1、经验 pass@k、cons@k、Verifier Top-1 |
| 过程 | 首错步骤准确率、PRM F1/AUC、解析失败率 |
| 成本 | 推理 Token、GPU 秒、P50/P95、超时率、单位正确答案成本 |
| 稳定性 | 截断、无最终答案、重复推理、语言混杂 |
| 鲁棒性 | 难度分层、组合外推、提示扰动、多随机种子 |
| 安全 | Reward Hacking、工具越权、敏感信息、推理日志权限 |

自然语言轨迹与参考轨迹的 BLEU/ROUGE 不是核心指标；同一问题可能有多条正确路径。最终答案正确也不能证明轨迹正确。


## 5．迁移到生产库

| 本章对象 | 生产库/组件 | 迁移时需要固定 |
|---|---|---|
| 问题、轨迹、步骤标签、奖励 | Hugging Face Datasets | Dataset Revision、Schema、Verifier Revision |
| Trace SFT | TRL `SFTTrainer` | Chat Template、Assistant/Completion Mask、Packing |
| 偏好训练 | TRL `DPOTrainer` | Reference Model、Beta、长度与截断 |
| ORM / PRM | TRL Reward/PRM Trainer | 标签粒度、校准集、聚合方式 |
| RLVR / GRPO | TRL `GRPOTrainer` | Reward Functions、Group Size、KL、Rollout Backend |
| 原理多样本生成 | `transformers.generate` | Sampling、Seed、Stop、输出切片 |
| 生产 Rollout/服务 | vLLM 或受支持后端 | Reasoning Parser、Structured Output、并发、Deadline |

TRL 可替代原理实现中的优化循环，但验证器定义仍属于任务契约；vLLM 提供批量采样和服务能力，预算、权限与输出治理仍由应用系统负责。


## 6．生产边界

1. 原始 Reasoning Trace、可验证证据和面向用户的解释是三种不同资产。
2. 默认只返回最终答案、关键假设和可复核证据；原始 CoT 不进入普通业务日志。
3. 若后端暴露 Reasoning 字段，必须独立授权、脱敏、限期保存，并与审计日志分离。
4. 代码执行、符号系统和工具环境运行在隔离沙箱，设置 CPU、内存、网络、时间和副作用限制。
5. Reward、Verifier 和 Parser 都可能被攻击或漂移；升级时运行隐藏集和对抗回归。
6. 超时或验证失败时降级为直接回答、证据包或人工处理；预算与权限边界不得在未授权情况下放宽。

### 6.1．参考资料
- [DeepSeek-R1](https://arxiv.org/abs/2501.12948)
- [Qwen3 官方说明](https://qwenlm.github.io/blog/qwen3/)
- [TRL 官方文档](https://huggingface.co/docs/trl/index)
- [vLLM Reasoning Outputs](https://docs.vllm.ai/en/stable/features/reasoning_outputs/)
- [Self-Consistency](https://arxiv.org/abs/2203.11171)
- [过程监督：Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)
